In [ ]:
#data for games against quad 2 team or better

###IMPORTANT### : MAKE A NEW MODEL FOR EACH SITUATION. NEUTRAL GAMES IN GENERAL/HOME OR AWAY IF ONE COLLEGE IS WITHIN 150 MILES, MAKE SURE IT IS
###FOR QUAD 1 AND 2 GAMES UNLESS A QUAD 3 TEAM MAKES IT, HAVE A MODEL FOR BOTH CONFERENCE AND NONCON JUST IN CASE WE RUN INTO THOSE GAMES

In [3]:
import pandas as pd
import re

# Load CSV
csv_path = r"C:\CSC450\TorvikMetricsInCon.csv"
df = pd.read_csv(csv_path)

# Remove any extra whitespace in column names
df.columns = df.columns.str.strip()

# Define weights
weights = {
    'Rk' : 0.044, #invert n
    'AdjOE': 0.10,   #y TIEBREAKER
    'AdjDE': 0.045,   #invert n
    'EFG%': 0.10, #y
    'EFGD%': 0.044, #invert n
    'TOR': 0.10, #invert y
    'TORD': 0.044, #n
    'ORB': 0.10, #y
    'DRB': 0.044, #invert n
    '3P%': 0.10, #y
    '3P%D':0.045, #invert n
    '2P%': 0.10, #y
    '2P%D': 0.044, #invert n
    'WAB': 0.045,#n
    'Barthag': 0.045 #n
}
# 6 y, 9 n, y = .10, n= 0.044

# Columns that are percentages
percent_cols = ['3P%', 'EFG%', '2P%D','EFGD%','ORB','DRB','3P%D','2P%','TOR','TORD']

# Columns where smaller is better
invert_cols = ['AdjDE', '2P%D','Rk','EFGD%','TOR','DRB','3P%D']

# Clean and convert all weighted columns
for col in weights.keys():
    # Remove % and commas
    df[col] = df[col].astype(str).str.replace('%','').str.replace(',','')
    df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Convert percentages to decimals
    if col in percent_cols:
        df[col] = df[col] / 100

# Invert metrics where smaller is better
for col in invert_cols:
    df[col] = df[col].max() - df[col]

# Normalize Efficiency Metrics
df['AdjOE'] = df['AdjOE'] / df['AdjOE'].max()
df['AdjDE'] = df['AdjDE'] / df['AdjDE'].max()

# Clean Team names
def clean_team_name(name):
    if pd.isna(name):
        return None
    name = str(name).strip()                  # remove leading/trailing spaces
    name = re.sub(r'^\d+\s*', '', name)      # remove any leading numbers
    name = re.sub(r'\s*\([AH]\)', '', name)  # remove (A) or (H)
    name = name.strip()
    return name

df['Team'] = df['Team'].apply(clean_team_name)

# Remove rows where Team is empty
df = df[df['Team'].notna()]

# Compute weighted score
df['WeightedScore'] = df[list(weights.keys())].dot(pd.Series(weights))

#Remove N/A
df = df.dropna(subset=['WeightedScore'])

# Sort descending
df_sorted = df.sort_values('WeightedScore', ascending=False).reset_index(drop=True)

# Show all rows in output
pd.set_option('display.max_rows', None)

# Display results
print(df_sorted[['Team', 'WeightedScore']])

# Optional: save to CSV
df_sorted[['Team', 'WeightedScore']].to_csv("RankedTeams_Cleaned.csv", index=False)


                       Team  WeightedScore
0                      Duke      16.937037
1                   Florida      16.890536
2                  Michigan      16.887491
3                   Arizona      16.831310
4                   Houston      16.656804
5                  Illinois      16.604806
6                  Nebraska      16.537035
7                Texas Tech      16.481640
8              Michigan St.      16.440623
9                    Purdue      16.410949
10              Connecticut      16.378022
11                 Virginia      16.278156
12                 Iowa St.      16.237682
13                Wisconsin      16.176004
14               St. John's      16.104996
15               Louisville      16.090222
16                 Arkansas      16.077032
17                Tennessee      16.047271
18                  Alabama      15.989959
19                   Kansas      15.904882
20               Vanderbilt      15.767760
21                 Ohio St.      15.737502
22         

In [4]:
# Show all rows
pd.set_option('display.max_rows', None)

# Now print the full DataFrame
print(df_sorted[['Team', 'WeightedScore']])

#maybe if within 4 points of opponent, use TORD stat to decide winner,else choose the higher WeightedScore

                       Team  WeightedScore
0                      Duke      16.937037
1                   Florida      16.890536
2                  Michigan      16.887491
3                   Arizona      16.831310
4                   Houston      16.656804
5                  Illinois      16.604806
6                  Nebraska      16.537035
7                Texas Tech      16.481640
8              Michigan St.      16.440623
9                    Purdue      16.410949
10              Connecticut      16.378022
11                 Virginia      16.278156
12                 Iowa St.      16.237682
13                Wisconsin      16.176004
14               St. John's      16.104996
15               Louisville      16.090222
16                 Arkansas      16.077032
17                Tennessee      16.047271
18                  Alabama      15.989959
19                   Kansas      15.904882
20               Vanderbilt      15.767760
21                 Ohio St.      15.737502
22         